# TN2 — Tầm nhìn DS-TCN 64 kênh, VÒNG XÁC NHẬN 4 fold

## Vòng sàng lọc cho kết quả gì

Vòng sàng lọc một fold, một seed. Tầm nhìn càng dài điểm càng tệ. Ba cấu hình
notebook này chạy tiếp:

| kernel | tầm nhìn | điểm `val_KL` | train_mse | train_pearson |
|---:|---:|---:|---:|---:|
| **5** | **121** | **0,8458** | 0,02127 | 0,5851 |
| **7** | **181** | **0,8176** | 0,02059 | 0,5921 |
| **9** | **241** | **0,8041** | 0,01939 | 0,6041 |

Tương quan tầm nhìn với điểm: **−0,983**.

Chỗ đáng chú ý: cùng lúc điểm tụt thì model **dự báo giỏi lên** — `train_mse`
giảm 13%, `train_pearson` tăng. Model càng giỏi dự báo càng chọn kênh dở.

Cơ chế có thể giải thích: tiêu chí chọn kênh là "ứng viên nào tự dự báo được
chính nó tốt nhất". Model tầm nhìn ngắn chỉ đoán giỏi sóng **thật sự tuần
hoàn**; model tầm nhìn dài đoán giỏi **mọi sóng trơn**, kể cả kênh nhiễu có cấu
trúc. Tầm nhìn ngắn hoạt động như một bộ lọc — dở đúng chỗ cần dở.

Đây mới là **giả thuyết**, chưa chứng minh.

## Vòng này chạy gì

Bỏ hai cấu hình tầm nhìn dài nhất (301, 361), giữ **ba cấu hình đầu** và chạy
**đủ bốn fold**, một seed.

| kernel | tầm nhìn | tham số | `val_KL` đã có |
|---:|---:|---:|---:|
| 5 | 121 | 38.105 | 0,8458 |
| 7 | 181 | 39.129 | 0,8176 |
| 9 | 241 | 40.153 | 0,8041 |

**Fold `val_KL` của cả ba đã chạy ở vòng sàng lọc.** Ô khôi phục kéo chúng về
từ Drive, nên `run_cv.py` sẽ in `đã có kết quả — bỏ qua` và chỉ train ba fold
còn lại. Ước lượng **1,5–2 giờ** thay vì gấp bốn.

Chạy xong, dòng `TONG` tự được ghi và `compare_cv` hiện `cv_score` đầy đủ.

## Vì sao vẫn chưa kết luận được sau vòng này

Một seed. `seed_std` của tám cấu hình TN1 trải từ 0,0007 tới 0,0108, mỗi kiến
trúc một khác — không mượn của nhau được. Muốn nói ba cấu hình khác nhau thật
thì phải chạy đủ ba seed cho chính chúng.

Vòng này trả lời câu hẹp hơn: **xu hướng thấy ở một fold có giữ nguyên trên đủ
bốn fold không?** Nếu `val_DF` — fold khó nhất — cho thứ tự ngược lại thì kết
luận từ `val_KL` là do fold chứ không do tầm nhìn.

## Mốc để đặt cạnh

| | tham số | tầm nhìn | cv_score |
|---|---:|---:|---:|
| DS-TCN-64 k3n4 no_norm do0.2 | 37.081 | **61** | 0,760878 ± 0,003095 *(3 seed)* |
| LSTM-352 | 1.502.713 | — | 0,756992 ± 0,004156 |
| DS-TCN-64 (TN1 gốc) | 56.281 | 253 | 0,742117 ± 0,000677 |

Dòng đầu là cùng kiến trúc, kernel 3, tầm nhìn 61 — điểm nối đầu bảng của thang
này.

## 1. Chuẩn bị Colab

In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


Tải mã nguồn.

In [2]:
!rm -rf /content/UWB_RADAR
!git clone -q --branch submission --single-branch https://github.com/quangminhho004-blip/UWB_RADAR.git /content/UWB_RADAR
%cd /content/UWB_RADAR
!python scripts/setup_colab.py

/content/UWB_RADAR

thư mục làm việc : /content/UWB_RADAR
commit đồ án     : 85ccc08
commit MobiVital : 4319731 (đã ghim)
GPU              : Tesla T4, 15360 MiB


Lấy `by_user/` và `windows/` từ Drive.

In [3]:
!python scripts/restore_processed_data_on_drive.py

by_user   : bung /content/drive/MyDrive/mobivital/by_user.tar ...
            12 tệp
windows   : bung /content/drive/MyDrive/mobivital/windows.tar.gz ...
            dev_cv 8 tệp, final_train có

2.5G	data/processed/by_user
503M	data/processed/windows


Khôi phục fold `val_KL` đã chạy ở vòng sàng lọc.

**Bước này quyết định notebook chạy 1,5 giờ hay 6 giờ.** Không có nó thì
`run_cv.py` train lại cả bốn fold. Phải thấy in ra ba dòng `giải nén`.

In [ ]:
# Khôi phục kết quả đã chạy trước khi phiên bị ngắt.
#
# Mỗi tệp nén chứa một bản runs/<thực nghiệm>/summary.csv của riêng nó. Giải
# hết vào cùng một thư mục runs/ thì tệp giải sau ĐÈ summary.csv của tệp trước.
# Mà sorted() xếp "..._a0_corr0.9..." đứng SAU "..._a0.9_corr0.9...", vì trong
# bảng mã ký tự "_" lớn hơn "." — nên bản ít dòng nhất lại là bản đè cuối cùng.
# Sửa: mỗi tệp nén giải vào một thư mục tạm riêng, gộp mọi dòng lại rồi mới ghi
# runs/summary.csv một lần. Thứ tự giải nén không còn ảnh hưởng gì nữa.
import csv, glob, os, shutil, subprocess, tempfile

MAU_ZIP = "/content/drive/MyDrive/mobivital/tn2_rf_*c64*.zip"

rows, seen = [], set()

def collect(summary_path):
    for r in csv.DictReader(open(summary_path)):
        key = (r.get("experiment"), r.get("run_id"))
        if key not in seen:
            seen.add(key)
            rows.append(r)

for f in sorted(glob.glob(MAU_ZIP)):
    tmp = tempfile.mkdtemp()
    subprocess.run(["unzip", "-oq", f, "-d", tmp], check=True)
    for s in glob.glob(tmp + "/*/summary.csv"):
        collect(s)
        os.remove(s)   # gộp xong thì bỏ, để bước chép dưới không đè nhau nữa
    # Checkpoint và curve.csv nằm trong thư mục riêng của từng lần chạy, tên
    # không trùng nhau, nên chép chồng lên runs/ là an toàn.
    for d in os.listdir(tmp):
        shutil.copytree(tmp + "/" + d, "runs/" + d, dirs_exist_ok=True)
    shutil.rmtree(tmp)

# Dòng đã sinh ra trong chính phiên này cũng phải giữ lại.
if os.path.exists("runs/summary.csv"):
    collect("runs/summary.csv")

if rows:
    # run_cv.py tra runs/summary.csv, còn tệp nén chỉ có runs/<thực nghiệm>/summary.csv
    cols = []
    for r in rows:
        for k in r:
            if k not in cols:
                cols.append(k)
    with open("runs/summary.csv", "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=cols, restval="")
        w.writeheader()
        w.writerows(rows)
print("khôi phục", len(rows), "dòng vào runs/summary.csv")

## 2. Kiểm ba bản cài đặt

**Đọc dòng cuối mỗi lệnh.** Phải là `TẤT CẢ ĐẠT`.

In [5]:
!python scripts/check_model.py --model ds_tcn --channels 64 \
    --kernel_size 5 --n_blocks 4 --dropout 0.2 --norm none --dropout_kind element

Kiểm model: ds_tcn

1. Hình dạng vào ra và giá trị
   vào (4, 200)  ->  ra (4, 25)
   ra đúng (4, 25)                                            đạt
   mọi giá trị hữu hạn                                        đạt

2. Gradient
   36/36 tham số nhận được gradient
   mọi tham số đều lan ngược tới                              đạt

3. Số tham số
   38105

4. Lưu và nạp lại state_dict
   nạp lại cho ra đúng đầu ra cũ                              đạt

5. Riêng TCN
   0 lớp BatchNorm1d, WeightNorm: False
   KHÔNG có lớp chuẩn hoá nào, đúng cấu hình đã chọn          đạt
   cũng không có WeightNorm                                   đạt

6. Loại dropout
   0 lớp Dropout1d (xoá cả kênh), 8 lớp Dropout (xoá từng phần tử)
   dùng nn.Dropout, xoá từng phần tử                          đạt

7. Tầm nhìn so với cửa sổ vào
   kernel 5, 4 khối -> tầm nhìn 121, cửa sổ vào 200
   CHÚ Ý: chỉ thấy 121/200 mẫu gần nhất, mất 40% đầu cửa sổ
   tầm nhìn ngắn hơn cửa sổ — có chủ ý, không phải lỗi        đạt

TẤT 

In [6]:
!python scripts/check_model.py --model ds_tcn --channels 64 \
    --kernel_size 7 --n_blocks 4 --dropout 0.2 --norm none --dropout_kind element

Kiểm model: ds_tcn

1. Hình dạng vào ra và giá trị
   vào (4, 200)  ->  ra (4, 25)
   ra đúng (4, 25)                                            đạt
   mọi giá trị hữu hạn                                        đạt

2. Gradient
   36/36 tham số nhận được gradient
   mọi tham số đều lan ngược tới                              đạt

3. Số tham số
   39129

4. Lưu và nạp lại state_dict
   nạp lại cho ra đúng đầu ra cũ                              đạt

5. Riêng TCN
   0 lớp BatchNorm1d, WeightNorm: False
   KHÔNG có lớp chuẩn hoá nào, đúng cấu hình đã chọn          đạt
   cũng không có WeightNorm                                   đạt

6. Loại dropout
   0 lớp Dropout1d (xoá cả kênh), 8 lớp Dropout (xoá từng phần tử)
   dùng nn.Dropout, xoá từng phần tử                          đạt

7. Tầm nhìn so với cửa sổ vào
   kernel 7, 4 khối -> tầm nhìn 181, cửa sổ vào 200
   CHÚ Ý: chỉ thấy 181/200 mẫu gần nhất, mất 9% đầu cửa sổ
   tầm nhìn ngắn hơn cửa sổ — có chủ ý, không phải lỗi        đạt

TẤT C

In [7]:
!python scripts/check_model.py --model ds_tcn --channels 64 \
    --kernel_size 9 --n_blocks 4 --dropout 0.2 --norm none --dropout_kind element

Kiểm model: ds_tcn

1. Hình dạng vào ra và giá trị
   vào (4, 200)  ->  ra (4, 25)
   ra đúng (4, 25)                                            đạt
   mọi giá trị hữu hạn                                        đạt

2. Gradient
   36/36 tham số nhận được gradient
   mọi tham số đều lan ngược tới                              đạt

3. Số tham số
   40153

4. Lưu và nạp lại state_dict
   nạp lại cho ra đúng đầu ra cũ                              đạt

5. Riêng TCN
   0 lớp BatchNorm1d, WeightNorm: False
   KHÔNG có lớp chuẩn hoá nào, đúng cấu hình đã chọn          đạt
   cũng không có WeightNorm                                   đạt

6. Loại dropout
   0 lớp Dropout1d (xoá cả kênh), 8 lớp Dropout (xoá từng phần tử)
   dùng nn.Dropout, xoá từng phần tử                          đạt

7. Tầm nhìn so với cửa sổ vào
   kernel 9, 4 khối -> tầm nhìn 241, cửa sổ vào 200
   phủ trọn cửa sổ                                            đạt

TẤT CẢ ĐẠT — bản cài đặt dùng được.


## 3. Chạy đủ 4 fold, một seed

Không có `--folds` nên chạy đủ bốn. Fold `val_KL` đã có sẽ bị bỏ qua — tìm dòng
`đã có kết quả 0.xxxx — bỏ qua, không train lại` để chắc ô khôi phục đã ăn.

Mỗi cấu hình khoảng **30 phút** cho ba fold còn lại.

**kernel 5 — tầm nhìn 121, 38.105 tham số, `val_KL` đã có 0,8458**

In [8]:
!python scripts/run_cv.py --experiment tn2_rf --model ds_tcn --channels 64 \
    --kernel_size 5 --n_blocks 4 --dropout 0.2 --norm none --dropout_kind element --seed 0

thực nghiệm tn2_rf  -> runs/tn2_rf/
cấu hình ds_tcn_c64_k5_n4_none_do0.2_dpel_mse_corr0.9_seed0
thiết bị  Tesla T4

----------------------------------------------------------
val_AB  train CDEFKL  chấm AB
210964 cửa sổ train
epoch  0  mse 0.05457  pearson 0.3421   0.4 phút
epoch  1  mse 0.02780  pearson 0.4885   0.8 phút
epoch  2  mse 0.02465  pearson 0.5270   1.2 phút
epoch  3  mse 0.02344  pearson 0.5432   1.6 phút
epoch  4  mse 0.02276  pearson 0.5499   1.9 phút
epoch  5  mse 0.02248  pearson 0.5542   2.3 phút
epoch  6  mse 0.02226  pearson 0.5570   2.7 phút
epoch  7  mse 0.02205  pearson 0.5601   3.1 phút
epoch  8  mse 0.02185  pearson 0.5623   3.5 phút
epoch  9  mse 0.02171  pearson 0.5650   3.9 phút
epoch 10  mse 0.02157  pearson 0.5666   4.3 phút
epoch 11  mse 0.02145  pearson 0.5683   4.6 phút
epoch 12  mse 0.02134  pearson 0.5697   5.0 phút
epoch 13  mse 0.02129  pearson 0.5718   5.4 phút
epoch 14  mse 0.02118  pearson 0.5726   5.8 phút
epoch 15  mse 0.02105  pearson 0.5737   

**kernel 7 — tầm nhìn 181, 39.129 tham số, `val_KL` đã có 0,8176**

In [9]:
!python scripts/run_cv.py --experiment tn2_rf --model ds_tcn --channels 64 \
    --kernel_size 7 --n_blocks 4 --dropout 0.2 --norm none --dropout_kind element --seed 0

thực nghiệm tn2_rf  -> runs/tn2_rf/
cấu hình ds_tcn_c64_k7_n4_none_do0.2_dpel_mse_corr0.9_seed0
thiết bị  Tesla T4

----------------------------------------------------------
val_AB  train CDEFKL  chấm AB
210964 cửa sổ train
epoch  0  mse 0.05217  pearson 0.3594   0.4 phút
epoch  1  mse 0.02653  pearson 0.4943   0.8 phút
epoch  2  mse 0.02399  pearson 0.5290   1.1 phút
epoch  3  mse 0.02291  pearson 0.5434   1.5 phút
epoch  4  mse 0.02237  pearson 0.5511   1.9 phút
epoch  5  mse 0.02197  pearson 0.5561   2.3 phút
epoch  6  mse 0.02166  pearson 0.5606   2.7 phút
epoch  7  mse 0.02146  pearson 0.5636   3.0 phút
epoch  8  mse 0.02124  pearson 0.5673   3.4 phút
epoch  9  mse 0.02106  pearson 0.5700   3.8 phút
epoch 10  mse 0.02090  pearson 0.5726   4.2 phút
epoch 11  mse 0.02080  pearson 0.5751   4.6 phút
epoch 12  mse 0.02064  pearson 0.5771   4.9 phút
epoch 13  mse 0.02052  pearson 0.5794   5.3 phút
epoch 14  mse 0.02040  pearson 0.5809   5.7 phút
epoch 15  mse 0.02034  pearson 0.5820   

**kernel 9 — tầm nhìn 241, 40.153 tham số, `val_KL` đã có 0,8041**

In [10]:
!python scripts/run_cv.py --experiment tn2_rf --model ds_tcn --channels 64 \
    --kernel_size 9 --n_blocks 4 --dropout 0.2 --norm none --dropout_kind element --seed 0

thực nghiệm tn2_rf  -> runs/tn2_rf/
cấu hình ds_tcn_c64_k9_n4_none_do0.2_dpel_mse_corr0.9_seed0
thiết bị  Tesla T4

----------------------------------------------------------
val_AB  train CDEFKL  chấm AB
210964 cửa sổ train
epoch  0  mse 0.05655  pearson 0.3605   0.4 phút
epoch  1  mse 0.02569  pearson 0.4930   0.8 phút
epoch  2  mse 0.02356  pearson 0.5295   1.2 phút
epoch  3  mse 0.02271  pearson 0.5444   1.5 phút
epoch  4  mse 0.02208  pearson 0.5536   1.9 phút
epoch  5  mse 0.02170  pearson 0.5603   2.3 phút
epoch  6  mse 0.02134  pearson 0.5660   2.7 phút
epoch  7  mse 0.02099  pearson 0.5709   3.1 phút
epoch  8  mse 0.02071  pearson 0.5752   3.5 phút
epoch  9  mse 0.02047  pearson 0.5795   3.8 phút
epoch 10  mse 0.02023  pearson 0.5820   4.2 phút
epoch 11  mse 0.02005  pearson 0.5854   4.6 phút
epoch 12  mse 0.01985  pearson 0.5875   5.0 phút
epoch 13  mse 0.01970  pearson 0.5914   5.4 phút
epoch 14  mse 0.01955  pearson 0.5928   5.7 phút
epoch 15  mse 0.01938  pearson 0.5942   

## 4. Cất kết quả

In [11]:
!python scripts/save_results.py tn2_rf --out tn2_rf_c64_4fold

runs/tn2_rf/  ->  runs/tn2_rf_c64_4fold.zip   (2.2 MB)
   15 dòng metric trong summary.csv

Bên trong:
        0  2026-09-07 21:27   tn2_rf/
        0  2026-09-07 19:01   tn2_rf/ds_tcn_c64_k11_n4_none_do0.2_dpel_mse_corr0.9_seed0_val_KL/
        0  2026-09-07 16:51   tn2_rf/ds_tcn_c64_k13_n4_none_do0.2_dpel_mse_corr0.9_seed0_val_KL/
        0  2026-09-07 19:09   tn2_rf/ds_tcn_c64_k5_n4_none_do0.2_dpel_mse_corr0.9_seed0_val_AB/
        0  2026-09-07 19:23   tn2_rf/ds_tcn_c64_k5_n4_none_do0.2_dpel_mse_corr0.9_seed0_val_CE/
        0  2026-09-07 19:37   tn2_rf/ds_tcn_c64_k5_n4_none_do0.2_dpel_mse_corr0.9_seed0_val_DF/
        0  2026-09-07 19:50   tn2_rf/ds_tcn_c64_k5_n4_none_do0.2_dpel_mse_corr0.9_seed0_val_KL/
        0  2026-09-07 20:02   tn2_rf/ds_tcn_c64_k7_n4_none_do0.2_dpel_mse_corr0.9_seed0_val_AB/
        0  2026-09-07 20:16   tn2_rf/ds_tcn_c64_k7_n4_none_do0.2_dpel_mse_corr0.9_seed0_val_CE/
        0  2026-09-07 20:29   tn2_rf/ds_tcn_c64_k7_n4_none_do0.2_dpel_mse_corr0.9_seed0_v

## 5. Bảng so

Giờ đã đủ bốn fold nên dòng `TONG` có, `compare_cv` hiện `cv_score` thật.

In [12]:
!python scripts/compare_cv.py --experiment tn2_rf


BẢNG 1 — cv_score, thực nghiệm tn2_rf
cấu hình                         tham số  seed    cv_mean  seed_std  fold_std   từng seed
--------------------------------------------------------------------------------------------------------------
ds_tcn_c64_k5_n4_none_do0.2_dpel_mse_corr0.9     38105     1   0.757855       N/A  0.093526   s0 0.7579
ds_tcn_c64_k7_n4_none_do0.2_dpel_mse_corr0.9     39129     1   0.743657       N/A  0.095832   s0 0.7437
ds_tcn_c64_k9_n4_none_do0.2_dpel_mse_corr0.9     40153     1   0.736970       N/A  0.089853   s0 0.7370

cv_mean  = trung bình cv_score của các seed. cv_score của một seed là
           trung bình điểm macro 4 fold; macro = trung bình theo NGƯỜI.
seed_std = dao động giữa các seed. Chênh lệch giữa hai cấu hình nhỏ hơn
           số này thì chưa kết luận được.
fold_std = dao động giữa 4 fold, trung bình trên các seed. Nói dữ liệu
           giữa các người khác nhau ra sao, KHÔNG dùng để so cấu hình.



## 6. Xu hướng còn giữ không

In điểm từng fold để xem thứ tự ba cấu hình có giống nhau ở mọi fold không. Nếu
`val_DF` — fold khó nhất — cho thứ tự ngược thì kết luận từ `val_KL` là do fold
chứ không do tầm nhìn.

In [13]:
import csv, re
RF = {5: 121, 7: 181, 9: 241}
rows = [r for r in csv.DictReader(open("runs/tn2_rf/summary.csv"))
        if "_c64_" in r["run_id"] and r["fold"] != "TONG"]
for r in sorted(rows, key=lambda r: (r["fold"], r["run_id"])):
    k = int(re.search(r"_k(\d+)_", r["run_id"]).group(1))
    print(" ", r["fold"], " kernel", k, " tầm nhìn", RF[k], " ", r["score_macro"])

  val_AB  kernel 5  tầm nhìn 121   0.7954657192439054
  val_AB  kernel 7  tầm nhìn 181   0.7892442252419967
  val_AB  kernel 9  tầm nhìn 241   0.7929070854897944
  val_CE  kernel 5  tầm nhìn 121   0.7805152598697644
  val_CE  kernel 7  tầm nhìn 181   0.7794378007378778
  val_CE  kernel 9  tầm nhìn 241   0.7860290347813774
  val_DF  kernel 5  tầm nhìn 121   0.6027316024192613
  val_DF  kernel 7  tầm nhìn 181   0.5803630552058399
  val_DF  kernel 9  tầm nhìn 241   0.581402128093564
  val_KL  kernel 5  tầm nhìn 121   0.8527084614489387
  val_KL  kernel 7  tầm nhìn 181   0.8255833006295739
  val_KL  kernel 9  tầm nhìn 241   0.7875405416645391


## 7. Ngắt phiên

In [ ]:
from google.colab import runtime
runtime.unassign()